# 03 RAG评测与生产化

**用途：** 用已有Top-K报告理解检索质量、拒答、同义词、增量索引和Chroma生产边界。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


## 1. Top-K不是拍脑袋

当前专项集比较K=1/3/5/8。K越大可能提高召回，也会增加噪声、Prompt长度、延迟和费用。应该同时观察来源命中、字段命中、失败数和距离，不只看“测试有没有报错”。

In [2]:
import re
reports = sorted((DAY1_ROOT / "reports").glob("topk_comparison_rag_observability_cases_*.md"))
check("存在Top-K报告", bool(reports), str(reports[-1]) if reports else "")
latest_report = reports[-1]
rows = []
for line in latest_report.read_text(encoding="utf-8").splitlines():
    if re.match(r"^\|\s*(1|3|5|8)\s*\|", line):
        parts = [part.strip() for part in line.strip("|").split("|")]
        rows.append({
            "K": int(parts[0]),
            "来源命中": parts[5],
            "工单关键词": parts[9],
            "低置信": int(parts[10]),
            "失败": int(parts[11]),
            "距离": parts[12],
        })
frame = show_table(rows)
failed_by_k = dict(zip(frame["K"], frame["失败"]))
check_equal("K=5专项失败数", failed_by_k[5], 0)
check("K=1召回不足可观察", failed_by_k[1] > failed_by_k[5])

[PASS] 存在Top-K报告 | D:\new things\项目1\day1\reports\topk_comparison_rag_observability_cases_20260727_134442.md


,K,来源命中,工单关键词,低置信,失败,距离
0,1,6/9 (66.7%),8/9 (88.9%),0,4,0.4747/0.6970/0.9554
1,3,9/9 (100.0%),8/9 (88.9%),0,1,0.4747/0.6970/0.9554
2,5,9/9 (100.0%),9/9 (100.0%),0,0,0.4747/0.6970/0.9554
3,8,9/9 (100.0%),9/9 (100.0%),0,0,0.4747/0.6970/0.9554


[PASS] K=5专项失败数 | actual=0, expected=0
[PASS] K=1召回不足可观察


{'检查项': 'K=1召回不足可观察', '状态': 'PASS', '说明': ''}

In [3]:
from tests.test_rag_components import FakeEmbeddings
from rag_components import build_index_fingerprint, fingerprint_changes

current = build_index_fingerprint(FakeEmbeddings(), chunk_size=500, chunk_overlap=80)
changed = dict(current)
changed["embedding_model"] = "another-embedding"
changed["chunk_size"] = 700
changes = fingerprint_changes(current, changed)
show_table([
    {"字段": key, "旧值": value["previous"], "新值": value["current"]}
    for key, value in changes.items()
])
check("Embedding变化被检测", "embedding_model" in changes)
check("Chunk变化被检测", "chunk_size" in changes)

D:\new things\项目1\day1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,字段,旧值,新值
0,chunk_size,500,700
1,embedding_model,BAAI/bge-small-zh-v1.5,another-embedding


[PASS] Embedding变化被检测
[PASS] Chunk变化被检测


{'检查项': 'Chunk变化被检测', '状态': 'PASS', '说明': ''}

## 必须回答的生产问题

- **检索不到正确文档怎么办？** 先区分资料不存在、切分失败、Embedding不匹配、查询表达不同和阈值过严；使用查询改写、同义词、BM25/混合检索、Rerank或转人工。
- **如何降低幻觉？** 距离阈值、证据不足拒答、Prompt限定、来源引用、结构化状态和工具执行边界。最终答案不能把检索内容当系统指令。
- **答案必须引用吗？** 企业知识回答应返回来源；库存、价格应引用工具结果而不是RAG文档。
- **只看测试通过率够吗？** 不够，还要看retrieval recall、source accuracy、faithfulness、answer relevance、拒答准确率、延迟和成本。
- **如何增量更新？** 计算文件hash和索引fingerprint，新增/变化文档重切分、删除旧chunk、写入新向量；Embedding或切分策略变化时全量重建。
- **如何处理“液压泵/主泵/泵总成”？** 查询规范化、领域同义词表、关键词召回与语义召回结合，并在评测集中加入表达变体。
- **库外问题怎么办？** 明确证据不足，追问、拒答或创建人工服务单，不能用模型常识冒充企业政策。
- **Chroma适合生产吗？** 适合作品集和单实例；规模增大后考虑带权限、备份、监控和水平扩展能力的Milvus、pgvector或托管向量库。

### 面试代码追问

1. 为什么当前专项K=5为0失败，但客户侧仍可能选择K=3？
2. L2距离阈值如何重新标定？
3. 索引fingerprint解决了什么兼容性风险？
4. 如何构建困难负例和同义词评测集？

### 参考答案

1. **为什么专项K=5零失败仍可能线上用K=3？** 当前结论只来自9条专项用例，K=5提高了召回，也会增加重复证据、无关内容、延迟和Prompt成本。应在更大代表性数据集上比较端到端正确率和拒答率；也可以先召回5条，再经阈值或Reranker压到3条进入模型。
2. **L2距离阈值如何标定？** 固定Embedding、归一化和距离度量，收集相关/不相关查询-文档对，观察两类距离分布，在验证集上根据业务代价选择阈值，并单独统计误召回与漏召回。更换Embedding后必须重新标定，不能沿用旧数字。
3. **fingerprint解决什么？** 它记录Embedding模型、切分参数和关键索引配置。启动或增量更新时发现不一致，就阻止把新查询配置与旧向量混用，并提示全量重建。
4. **困难负例和同义词怎么构建？** 正例加入“液压泵/主泵/泵总成”等表达变体；困难负例选择品牌相同但机型、年份或件号不同的文档；再加入库外问题、拼写错误和信息冲突，分别评估召回、来源和拒答。

**代码落点：** `evaluate_cases.py`、`tests/rag_observability_cases.jsonl`、`rag_components.py`和`reports/topk_comparison_*.md`。

In [4]:
check("生产化知识已覆盖", len(rows) == 4, "已读取K=1/3/5/8四组结果")

[PASS] 生产化知识已覆盖 | 已读取K=1/3/5/8四组结果


{'检查项': '生产化知识已覆盖', '状态': 'PASS', '说明': '已读取K=1/3/5/8四组结果'}